# LeetCode 1387 — Sort Integers by The Power Value bằng Python

Link đề: https://leetcode.com/problems/sort-integers-by-the-power-value/ · Độ khó: **Medium** · Chủ đề: Dynamic Programming, Memoization, Sorting

Skeleton LeetCode:

```python
class Solution:
    def getKth(self, lo: int, hi: int, k: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Hiểu định nghĩa **power** (số bước Collatz về 1) và tính đúng, không lệch một.
2. Dùng `sorted` với **khóa tuple** `(power, x)` để sắp theo nhiều tiêu chí.
3. Nhận ra các dãy Collatz **chia sẻ đuôi** ⇒ dùng **memoization** (`dict`, `functools.lru_cache`, hoặc ghi nhớ lặp theo đường đi) để không tính lại.
4. Biết chọn phần tử thứ `k` bằng **heap** (`heapq.nsmallest`) thay vì sắp toàn bộ khi `k` nhỏ.
5. So sánh chi phí thực tế giữa tính trực tiếp và có nhớ trên ràng buộc của đề.

## 2. Đề bài & ví dụ

**Power** của số nguyên `x` là số bước để biến `x` thành `1` theo quy tắc:
- nếu `x` chẵn: `x = x / 2`
- nếu `x` lẻ: `x = 3·x + 1`

Ví dụ power của 3 là 7: `3 → 10 → 5 → 16 → 8 → 4 → 2 → 1`.

Cho `lo`, `hi`, `k`. Sắp mọi số nguyên trong `[lo, hi]` theo power **tăng dần**; nếu power bằng nhau thì theo **giá trị tăng dần**. Trả về số thứ `k` (1-based) trong dãy đã sắp.

- **Ví dụ 1:** `lo = 12, hi = 15, k = 2` → `13`. Power: 12 → 9, 13 → 9, 14 → 17, 15 → 17. Dãy sắp `[12, 13, 14, 15]`.
- **Ví dụ 2:** `lo = 7, hi = 11, k = 4` → `7`. Power `[16, 3, 19, 6, 14]` cho `[7..11]`; dãy sắp `[8, 10, 11, 7, 9]`.

**Ràng buộc:** `1 <= lo <= hi <= 1000`; `1 <= k <= hi − lo + 1`. Đề đảm bảo mọi `x` trong khoảng đều về được 1 và power vừa số nguyên 32 bit.

In [1]:
import heapq
import random
import time
from functools import lru_cache

EXAMPLES = [(12, 15, 2, 13), (7, 11, 4, 7)]
for lo, hi, k, want in EXAMPLES:
    print(f"lo={lo}, hi={hi}, k={k} -> mong đợi {want}")

lo=12, hi=15, k=2 -> mong đợi 13
lo=7, hi=11, k=4 -> mong đợi 7


## 3. Kiến thức nền

### 3.1 Vòng `while`, phép chia nguyên `//`, toán tử `%`

`x % 2 == 0` kiểm tra chẵn; dùng `x // 2` (chia nguyên) thay vì `x / 2` (ra số thực `float`).

In [2]:
x = 6
print(x % 2 == 0, x // 2, x / 2, type(x / 2).__name__)
seq = [3]
while seq[-1] != 1:
    v = seq[-1]
    seq.append(v // 2 if v % 2 == 0 else 3 * v + 1)
print("dãy Collatz của 3:", seq, "| số bước =", len(seq) - 1)

True 3 3.0 float
dãy Collatz của 3: [3, 10, 5, 16, 8, 4, 2, 1] | số bước = 7


### 3.2 `sorted` với khóa tuple

Python so sánh tuple **theo từng phần tử từ trái sang**: `(9, 12) < (9, 13) < (17, 14)`. Vì vậy `key=lambda x: (power(x), x)` sắp theo power rồi theo giá trị.

In [3]:
pairs = [(17, 14), (9, 13), (17, 15), (9, 12)]
print(sorted(pairs))
print(sorted([15, 14, 13, 12], key=lambda x: (x % 3, x)))

[(9, 12), (9, 13), (17, 14), (17, 15)]
[12, 15, 13, 14]


### 3.3 Memoization với `dict` và `functools.lru_cache`

Ghi nhớ kết quả hàm theo tham số để lần sau trả ngay. `@lru_cache(maxsize=None)` tự làm điều này cho hàm thuần (cùng đầu vào → cùng đầu ra). `cache_info()` cho biết số lần trúng/trượt.

In [4]:
@lru_cache(maxsize=None)
def slow_square(n):
    return n * n

for v in [3, 4, 3, 3]:
    slow_square(v)
print(slow_square.cache_info())

CacheInfo(hits=2, misses=2, maxsize=None, currsize=2)


### 3.4 `heapq.nsmallest` — lấy `k` phần tử nhỏ nhất

`heapq.nsmallest(k, iterable, key=...)` trả `k` phần tử nhỏ nhất theo khóa (đã sắp), chi phí `O(N log k)` — tốt hơn sắp toàn bộ khi `k ≪ N`.

In [5]:
data = [5, 1, 9, 3, 7, 2]
print(heapq.nsmallest(3, data))
print(heapq.nsmallest(2, ["bb", "a", "ccc"], key=len))

[1, 2, 3]
['a', 'bb']


## 4. Ý tưởng nền tảng / trực giác

Tính tay Ví dụ 2 (`[7..11]`):

| x | dãy Collatz | power |
|---|---|---|
| 7 | 7 → 22 → 11 → 34 → 17 → 52 → 26 → 13 → 40 → 20 → 10 → 5 → 16 → 8 → 4 → 2 → 1 | 16 |
| 8 | 8 → 4 → 2 → 1 | 3 |
| 9 | 9 → 28 → 14 → 7 → … (như trên) | 3 + 16 = 19 |
| 10 | 10 → 5 → 16 → 8 → 4 → 2 → 1 | 6 |
| 11 | 11 → 34 → … (đuôi của 7) | 14 |

Sắp theo `(power, x)`: `(3,8), (6,10), (14,11), (16,7), (19,9)` ⇒ phần tử thứ 4 là **7**.

Quan sát then chốt: dãy của 9 đi qua 7, dãy của 7 đi qua 11, 10... ⇒ **power(x) = 1 + power(bước kế tiếp)**. Nếu đã biết power của 7 thì power của 9 = 3 + 16, không cần đi lại.

## 5. Cách 1 — Brute Force / baseline: tính power trực tiếp rồi sắp xếp

In [6]:
def power_direct(x):
    steps = 0
    while x != 1:
        x = x // 2 if x % 2 == 0 else 3 * x + 1
        steps += 1
    return steps

def kth_direct(lo, hi, k):
    nums = sorted(range(lo, hi + 1), key=lambda x: (power_direct(x), x))
    return nums[k - 1]

print("power(3) =", power_direct(3), "| power 7..11 =", [power_direct(x) for x in range(7, 12)])
for lo, hi, k, want in EXAMPLES:
    print(kth_direct(lo, hi, k), "(mong đợi", want, ")")

power(3) = 7 | power 7..11 = [16, 3, 19, 6, 14]
13 (mong đợi 13 )
7 (mong đợi 7 )


**Độ phức tạp:** gọi `N = hi − lo + 1 ≤ 1000`, `P` = power lớn nhất (với `x ≤ 1000`, `P = 178` tại `x = 871`). Thời gian `O(N·P + N log N)` ≈ `1,8·10^5` bước — **đủ nhanh** cho ràng buộc. Bộ nhớ `O(N)`.

**Nút thắt (khi ràng buộc lớn hơn hoặc nhiều truy vấn):** các dãy dùng chung đuôi rất nhiều nhưng ta tính lại từ đầu mỗi lần. Đo tổng số bước thực sự thực hiện:

In [7]:
total_steps = sum(power_direct(x) for x in range(1, 1001))
print("tổng số bước khi tính trực tiếp cho 1..1000:", total_steps)
print("power lớn nhất trong 1..1000:", max((power_direct(x), x) for x in range(1, 1001)))

tổng số bước khi tính trực tiếp cho 1..1000: 59542
power lớn nhất trong 1..1000: (178, 871)


## 6. Ý tưởng thuật toán / bước đột phá

**Insight (truy hồi):**

- `power(1) = 0`
- `power(x) = 1 + power(x // 2)` nếu `x` chẵn
- `power(x) = 1 + power(3x + 1)` nếu `x` lẻ

Mỗi giá trị chỉ cần tính **một lần** rồi ghi nhớ ⇒ tổng công bằng số **giá trị phân biệt** xuất hiện trong mọi dãy, thay vì tổng độ dài các dãy.

**Lưu ý:** dãy có thể vượt xa `hi` (ví dụ 27 lên tới 9232; với `x ≤ 1000` đỉnh cao nhất là 250504) ⇒ dùng **`dict`**, không dùng mảng cỡ `hi`.

**Quy tắc "nếu… thì…":**
- Nếu `x` đã có trong memo ⇒ trả ngay.
- Nếu chưa ⇒ đi tiếp một bước, cộng 1, và **ghi lại**.

Sau khi có power, bài toán còn lại là **sắp theo khóa `(power, x)`** và lấy phần tử thứ `k`.

Kiểm chứng nhanh hai nhận xét trên bằng số liệu: (1) bao nhiêu giá trị trung gian **vượt quá 1000** — lý do phải dùng `dict`; (2) phân bố power trong `1..1000` — có rất nhiều số **hòa power**, nên tiêu chí phụ `x` thực sự quan trọng.

In [8]:
from collections import Counter

visited = set()
for x in range(1, 1001):
    y = x
    while y not in visited:
        visited.add(y)
        if y == 1:
            break
        y = y // 2 if y % 2 == 0 else 3 * y + 1
over = [v for v in visited if v > 1000]
print(f"giá trị phân biệt: {len(visited)}, trong đó > 1000: {len(over)}, lớn nhất: {max(visited)}")

freq = Counter(power_direct(x) for x in range(1, 1001))
top = freq.most_common(5)
print("5 giá trị power phổ biến nhất (power: số lượng):", top)
print("số nhóm power có từ 2 số trở lên:", sum(1 for c in freq.values() if c >= 2), "/", len(freq))

giá trị phân biệt: 2228, trong đó > 1000: 1228, lớn nhất: 250504
5 giá trị power phổ biến nhất (power: số lượng): [(28, 30), (33, 25), (30, 23), (25, 23), (20, 22)]
số nhóm power có từ 2 số trở lên: 123 / 142


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Memo đệ quy với `lru_cache`

Độ sâu đệ quy tối đa = power lớn nhất (178 với `x ≤ 1000`) ⇒ an toàn với giới hạn đệ quy mặc định.

In [9]:
@lru_cache(maxsize=None)
def power_memo(x):
    if x == 1:
        return 0
    return 1 + power_memo(x // 2 if x % 2 == 0 else 3 * x + 1)

def kth_memo(lo, hi, k):
    return sorted(range(lo, hi + 1), key=lambda x: (power_memo(x), x))[k - 1]

for lo, hi, k, want in EXAMPLES:
    print(kth_memo(lo, hi, k), "(mong đợi", want, ")")
power_memo.cache_clear()
_ = [power_memo(x) for x in range(1, 1001)]
print("số giá trị phân biệt đã ghi nhớ cho 1..1000:", power_memo.cache_info().currsize)

13 (mong đợi 13 )
7 (mong đợi 7 )
số giá trị phân biệt đã ghi nhớ cho 1..1000: 2228


So sánh: tính trực tiếp tốn 59.542 bước cho `1..1000`, còn memo chỉ cần **~2.200** lần tính (mỗi giá trị phân biệt một lần).

Code nộp LeetCode (Cách 2):

In [10]:
class Solution:
    def getKth(self, lo: int, hi: int, k: int) -> int:
        @lru_cache(maxsize=None)
        def power(x):
            if x == 1:
                return 0
            return 1 + power(x // 2 if x % 2 == 0 else 3 * x + 1)
        return sorted(range(lo, hi + 1), key=lambda x: (power(x), x))[k - 1]

print([Solution().getKth(lo, hi, k) for lo, hi, k, _ in EXAMPLES])

[13, 7]


**Độ phức tạp Cách 2:** `O(D + N log N)` với `D` = số giá trị phân biệt trong các dãy (≈ 2.200 cho cả `1..1000`); bộ nhớ `O(D)`.

### 7.2 Cách 3 — Memo lặp: đi tới khi gặp giá trị đã biết rồi điền ngược

Không dùng đệ quy (hữu ích nếu ràng buộc lớn khiến dãy rất dài):
1. Đi theo dãy, đẩy các giá trị **chưa biết** vào `path`, dừng khi gặp `x` đã có trong `memo`.
2. Duyệt `path` ngược lại, mỗi giá trị có power = power của giá trị sau nó + 1.

In [11]:
MEMO = {1: 0}

def power_iter(x):
    path = []
    while x not in MEMO:
        path.append(x)
        x = x // 2 if x % 2 == 0 else 3 * x + 1
    p = MEMO[x]
    for v in reversed(path):
        p += 1
        MEMO[v] = p
    return MEMO[path[0]] if path else MEMO[x]

def kth_iter(lo, hi, k):
    return sorted(range(lo, hi + 1), key=lambda x: (power_iter(x), x))[k - 1]

for lo, hi, k, want in EXAMPLES:
    print(kth_iter(lo, hi, k), "(mong đợi", want, ")")

13 (mong đợi 13 )
7 (mong đợi 7 )


Code nộp LeetCode (Cách 3):

In [12]:
class Solution:
    memo = {1: 0}                      # dùng chung giữa các test (thuộc tính lớp)

    def getKth(self, lo: int, hi: int, k: int) -> int:
        memo = self.memo
        def power(x):
            path = []
            while x not in memo:
                path.append(x)
                x = x // 2 if x % 2 == 0 else 3 * x + 1
            p = memo[x]
            for v in reversed(path):
                p += 1
                memo[v] = p
            return p
        return sorted(range(lo, hi + 1), key=lambda x: (power(x), x))[k - 1]

print([Solution().getKth(lo, hi, k) for lo, hi, k, _ in EXAMPLES])

[13, 7]


Lưu ý: khi `path` rỗng (x đã biết), vòng `for` không chạy và `p = memo[x]` — vẫn đúng. Memo ở mức **lớp** được tái dùng qua mọi test case của LeetCode (tất cả cùng một tiến trình).

**Độ phức tạp Cách 3:** như Cách 2, `O(D + N log N)`; không đệ quy.

### 7.3 Cách 4 — Bảng tính sẵn + `heapq.nsmallest(k)`

Vì `hi ≤ 1000`, có thể tính sẵn power cho mọi `1..1000` **một lần** (mức module/lớp). Mỗi truy vấn chỉ cần chọn `k` phần tử nhỏ nhất theo `(power, x)` bằng heap: `O(N log k)`.

In [13]:
POWER_TABLE = [0] * 1001
for x in range(1, 1001):
    POWER_TABLE[x] = power_iter(x)

def kth_heap(lo, hi, k):
    return heapq.nsmallest(k, range(lo, hi + 1), key=lambda x: (POWER_TABLE[x], x))[-1]

for lo, hi, k, want in EXAMPLES:
    print(kth_heap(lo, hi, k), "(mong đợi", want, ")")

13 (mong đợi 13 )
7 (mong đợi 7 )


Code nộp LeetCode (Cách 4):

In [14]:
def _power(x, memo={1: 0}):
    path = []
    while x not in memo:
        path.append(x)
        x = x // 2 if x % 2 == 0 else 3 * x + 1
    p = memo[x]
    for v in reversed(path):
        p += 1
        memo[v] = p
    return p

_TABLE = [0] + [_power(x) for x in range(1, 1001)]     # tính một lần khi nạp module

class Solution:
    def getKth(self, lo: int, hi: int, k: int) -> int:
        return heapq.nsmallest(k, range(lo, hi + 1), key=lambda x: (_TABLE[x], x))[-1]

print([Solution().getKth(lo, hi, k) for lo, hi, k, _ in EXAMPLES])

[13, 7]


**Độ phức tạp Cách 4:** tiền xử lý `O(D)` một lần; mỗi truy vấn `O(N log k)`; bộ nhớ `O(D + 1000)`.

## 8. TRACE từng bước

**Trace 1:** tính power(9) bằng Cách 3 khi memo đã có power của các số từ dãy 7 (mô phỏng thứ tự tính `7, 8, 9`). Hàm `trace_power_iter` in `path` gom được và quá trình điền ngược.

In [15]:
def trace_power_iter(x, memo):
    start, path = x, []
    while x not in memo:
        path.append(x)
        x = x // 2 if x % 2 == 0 else 3 * x + 1
    print(f"power({start}): path chưa biết = {path}, dừng tại {x} (memo[{x}] = {memo[x]})")
    p = memo[x]
    for v in reversed(path):
        p += 1
        memo[v] = p
        print(f"   memo[{v}] = {p}")
    return memo[start]

memo_demo = {1: 0}
for v in [7, 8, 9]:
    print("=>", trace_power_iter(v, memo_demo))

power(7): path chưa biết = [7, 22, 11, 34, 17, 52, 26, 13, 40, 20, 10, 5, 16, 8, 4, 2], dừng tại 1 (memo[1] = 0)
   memo[2] = 1
   memo[4] = 2
   memo[8] = 3
   memo[16] = 4
   memo[5] = 5
   memo[10] = 6
   memo[20] = 7
   memo[40] = 8
   memo[13] = 9
   memo[26] = 10
   memo[52] = 11
   memo[17] = 12
   memo[34] = 13
   memo[11] = 14
   memo[22] = 15
   memo[7] = 16
=> 16
power(8): path chưa biết = [], dừng tại 8 (memo[8] = 3)
=> 3
power(9): path chưa biết = [9, 28, 14], dừng tại 7 (memo[7] = 16)
   memo[14] = 17
   memo[28] = 18
   memo[9] = 19
=> 19


Quan sát: `power(9)` chỉ đi 3 bước `9 → 28 → 14 → 7` rồi dùng `memo[7] = 16`; `power(8)` có sẵn trong memo (8 nằm trên dãy của 7) nên `path` rỗng và trả ngay.

**Trace 2:** bảng sắp theo khóa `(power, x)` cho Ví dụ 1 và 2:

| x | 12 | 13 | 14 | 15 | | 7 | 8 | 9 | 10 | 11 |
|---|---|---|---|---|---|---|---|---|---|---|
| power | 9 | 9 | 17 | 17 | | 16 | 3 | 19 | 6 | 14 |

In [16]:
def trace_sort(lo, hi, k):
    keyed = [(power_iter(x), x) for x in range(lo, hi + 1)]
    print("trước khi sắp:", keyed)
    keyed.sort()
    for i, (p, x) in enumerate(keyed, start=1):
        mark = "  <== phần tử thứ k" if i == k else ""
        print(f"  #{i}: x={x:3d}, power={p:3d}{mark}")
    return keyed[k - 1][1]

for lo, hi, k, want in EXAMPLES:
    print("kết quả:", trace_sort(lo, hi, k), "| mong đợi:", want)

trước khi sắp: [(9, 12), (9, 13), (17, 14), (17, 15)]
  #1: x= 12, power=  9
  #2: x= 13, power=  9  <== phần tử thứ k
  #3: x= 14, power= 17
  #4: x= 15, power= 17
kết quả: 13 | mong đợi: 13
trước khi sắp: [(16, 7), (3, 8), (19, 9), (6, 10), (14, 11)]
  #1: x=  8, power=  3
  #2: x= 10, power=  6
  #3: x= 11, power= 14
  #4: x=  7, power= 16  <== phần tử thứ k
  #5: x=  9, power= 19
kết quả: 7 | mong đợi: 7


## 9. Code nộp LeetCode

In [17]:
class Solution:
    memo = {1: 0}

    def getKth(self, lo: int, hi: int, k: int) -> int:
        memo = self.memo

        def power(x):
            path = []
            while x not in memo:                       # đi tới giá trị đã biết
                path.append(x)
                x = x // 2 if x % 2 == 0 else 3 * x + 1
            p = memo[x]
            for v in reversed(path):                   # điền ngược
                p += 1
                memo[v] = p
            return p

        return sorted(range(lo, hi + 1), key=lambda x: (power(x), x))[k - 1]

for lo, hi, k, want in EXAMPLES:
    got = Solution().getKth(lo, hi, k)
    print(got, "OK" if got == want else "SAI")

13 OK
7 OK


## 10. Vì sao thuật toán đúng

1. **Truy hồi đúng:** theo định nghĩa, từ `x ≠ 1` ta làm đúng một bước để tới `f(x)` (`x/2` hoặc `3x+1`), rồi cần `power(f(x))` bước nữa ⇒ `power(x) = 1 + power(f(x))`. Đề đảm bảo dãy dừng ở 1 nên truy hồi kết thúc.
2. **Memo đúng (Cách 3):** bất biến: `memo[v]` luôn là power thật của `v`. Khi điền ngược `path = [x0, x1, ..., xm]` với `f(xm) = y` đã biết: `power(xm) = memo[y] + 1`, rồi `power(x_{m−1}) = power(xm) + 1`, ... — đúng theo (1).
3. **Sắp xếp đúng:** so sánh tuple `(power, x)` đúng bằng thứ tự đề yêu cầu (power trước, giá trị sau). Phần tử thứ `k` (1-based) là chỉ số `k − 1`.
4. **Heap (Cách 4):** `nsmallest(k, ...)` trả `k` phần tử nhỏ nhất **theo đúng thứ tự** ⇒ phần tử cuối là phần tử thứ `k`.

## 11. Phân tích Time / Space

Gọi `N = hi − lo + 1 ≤ 1000`, `P` = power lớn nhất trong khoảng (≤ 178), `D` = số giá trị phân biệt xuất hiện trong các dãy (≈ 2.200 cho `1..1000`).

| Cách | Time | Space |
|---|---|---|
| 1. Trực tiếp + sort | `O(N·P + N log N)` (≈ 6·10^4 bước cho `1..1000`) | `O(N)` |
| 2. `lru_cache` + sort | `O(D + N log N)` | `O(D)` + đệ quy `O(P)` |
| 3. Memo lặp + sort | `O(D + N log N)` | `O(D)` |
| 4. Bảng sẵn + heap | tiền xử lý `O(D)`; truy vấn `O(N log k)` | `O(D)` |

Với ràng buộc nhỏ của đề, cả bốn cách đều chạy tức thì; memo trở nên quan trọng khi `hi` lớn (ví dụ `10^6`) hoặc có rất nhiều truy vấn.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [18]:
METHODS = {
    "direct": kth_direct,
    "memo": kth_memo,
    "iter": kth_iter,
    "heap": kth_heap,
    "submit": lambda lo, hi, k: Solution().getKth(lo, hi, k),
}
edge = [
    (1, 1, 1, 1),          # khoảng một phần tử
    (1, 1000, 1, 1),       # power(1) = 0 nhỏ nhất
    (1, 1000, 1000, 871),  # power lớn nhất là 178 tại 871
    (2, 2, 1, 2),
    (12, 15, 4, 15),       # hòa power: lấy giá trị lớn hơn ở cuối
    (10, 20, 5, 13),
    (1000, 1000, 1, 1000),
]
for name, fn in METHODS.items():
    for lo, hi, k, want in EXAMPLES + edge:
        got = fn(lo, hi, k)
        assert got == want, (name, lo, hi, k, got, want)
print("Tất cả", len(METHODS), "cách đúng trên", len(EXAMPLES) + len(edge), "test")

Tất cả 5 cách đúng trên 9 test


### 12.2 Đối chiếu ngẫu nhiên (seed cố định, đúng ràng buộc)

In [19]:
rng = random.Random(1387)
for _ in range(400):
    lo = rng.randint(1, 1000)
    hi = rng.randint(lo, min(1000, lo + rng.choice([0, 5, 50, 1000])))
    k = rng.randint(1, hi - lo + 1)
    res = {name: fn(lo, hi, k) for name, fn in METHODS.items()}
    assert len(set(res.values())) == 1, (lo, hi, k, res)
print("400 truy vấn ngẫu nhiên: mọi cách khớp nhau")

# kiểm tra bảng power với cách trực tiếp cho toàn bộ 1..1000
assert all(POWER_TABLE[x] == power_direct(x) for x in range(1, 1001))
print("POWER_TABLE khớp power_direct trên 1..1000")

400 truy vấn ngẫu nhiên: mọi cách khớp nhau
POWER_TABLE khớp power_direct trên 1..1000


### 12.3 Đo thời gian: truy vấn lớn nhất `[1, 1000]` lặp 200 lần

In [20]:
for name in ["direct", "memo", "iter", "heap", "submit"]:
    fn = METHODS[name]
    t0 = time.perf_counter()
    for k in range(1, 1001, 5):
        fn(1, 1000, k)
    print(f"{name:7s}: {time.perf_counter() - t0:.3f}s cho 200 truy vấn [1, 1000]")

direct : 0.795s cho 200 truy vấn [1, 1000]
memo   : 0.038s cho 200 truy vấn [1, 1000]
iter   : 0.060s cho 200 truy vấn [1, 1000]
heap   : 0.066s cho 200 truy vấn [1, 1000]


submit : 0.058s cho 200 truy vấn [1, 1000]


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Đếm số **phần tử** của dãy thay vì số **bước**

In [21]:
def wrong_power_count_terms(x):
    cnt = 1                       # SAI: đếm luôn cả x ban đầu
    while x != 1:
        x = x // 2 if x % 2 == 0 else 3 * x + 1
        cnt += 1
    return cnt

print("power(3): sai =", wrong_power_count_terms(3), "| đúng =", power_direct(3))
print("power(1): sai =", wrong_power_count_terms(1), "| đúng =", power_direct(1))

power(3): sai = 8 | đúng = 7
power(1): sai = 1 | đúng = 0


**Vì sao sai:** đề định nghĩa power là số **bước biến đổi**: `3` có 8 số trong dãy nhưng chỉ 7 bước; `power(1) = 0`. Nếu lệch đều thì thứ tự sắp **không đổi** (cộng 1 cho mọi số) — lỗi này có thể "may mắn" vẫn AC, nhưng sẽ sai ngay khi đề hỏi giá trị power.

**Code đúng:** bắt đầu `steps = 0`, tăng sau mỗi lần biến đổi.

### Lỗi 2 — Lưu `dict` theo **power** (power → x) làm mất các số hòa power

In [22]:
def wrong_dict_by_power(lo, hi, k):
    by_power = {}
    for x in range(lo, hi + 1):
        by_power[power_direct(x)] = x          # SAI: số sau ghi đè số trước cùng power
    order = [by_power[p] for p in sorted(by_power)]
    return order[k - 1] if k - 1 < len(order) else None

for lo, hi, k, want in EXAMPLES:
    print(f"lo={lo}, hi={hi}, k={k}: sai = {wrong_dict_by_power(lo, hi, k)} | đúng = {want}")

lo=12, hi=15, k=2: sai = 15 | đúng = 13
lo=7, hi=11, k=4: sai = 7 | đúng = 7


**Vì sao sai:** 12 và 13 cùng power 9 ⇒ `13` ghi đè `12`; dãy chỉ còn `[13, 15]` ⇒ phần tử thứ 2 là 15 thay vì 13. Khóa của dict phải **duy nhất** — power thì không.

**Code đúng:** sắp danh sách số theo khóa tuple `(power(x), x)`.

### Lỗi 3 — Quên `k` là 1-based

In [23]:
def wrong_index(lo, hi, k):
    return sorted(range(lo, hi + 1), key=lambda x: (power_direct(x), x))[k]   # SAI: phải là k - 1

for lo, hi, k, want in EXAMPLES:
    print(f"sai = {wrong_index(lo, hi, k)} | đúng = {want}")
try:
    wrong_index(12, 15, 4)
except IndexError as e:
    print("k = hi - lo + 1 -> IndexError:", e)

sai = 14 | đúng = 13
sai = 9 | đúng = 7
k = hi - lo + 1 -> IndexError: list index out of range


**Vì sao sai:** "phần tử thứ `k`" theo cách nói thông thường là chỉ số `k − 1` trong list Python. Dùng `[k]` trả phần tử kế tiếp (sai), và văng `IndexError` khi `k` bằng số phần tử.

**Code đúng:** `[k - 1]`.

### Lỗi 4 — Dùng `range(lo, hi)` bỏ sót `hi`

In [24]:
def wrong_range(lo, hi, k):
    nums = sorted(range(lo, hi), key=lambda x: (power_direct(x), x))   # SAI: thiếu hi
    return nums[k - 1] if k - 1 < len(nums) else None

for lo, hi, k, want in [(7, 11, 3, 11), (12, 15, 4, 15), (5, 5, 1, 5)]:
    print(f"[{lo},{hi}], k={k}: sai = {wrong_range(lo, hi, k)} | đúng = {kth_memo(lo, hi, k)}")

[7,11], k=3: sai = 7 | đúng = 11
[12,15], k=4: sai = None | đúng = 15
[5,5], k=1: sai = None | đúng = 5


**Vì sao sai:** đoạn `[lo, hi]` **bao gồm** `hi`, trong khi `range(lo, hi)` dừng ở `hi − 1`. Thiếu một số có thể làm đổi phần tử thứ `k` hoặc khiến chỉ số vượt ngoài.

**Code đúng:** `range(lo, hi + 1)`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Sắp theo tiêu chí A, hòa thì theo B"** ⇒ `sorted(..., key=lambda x: (A(x), B(x)))`; muốn giảm dần một tiêu chí số thì đổi dấu (`-A(x)`).
- **Hàm đệ quy có đầu vào lặp lại nhiều** (Collatz, Fibonacci, đếm đường đi) ⇒ `@lru_cache(maxsize=None)` là cách nhanh nhất để thêm memo.
- Giá trị trung gian có thể **vượt xa** biên đầu vào ⇒ memo bằng `dict`, không bằng mảng cỡ `hi`.
- **Lấy phần tử thứ `k`** với `k` nhỏ ⇒ `heapq.nsmallest(k, ...)`; với `k` bất kỳ thì sort `O(N log N)` vẫn ổn khi `N ≤ 10^5`.
- Ràng buộc nhỏ cố định (`≤ 1000`) ⇒ có thể **tính sẵn** bảng một lần ở mức module/lớp.

## 15. Biến thể của bài

1. **Số trong `[1, N]` có power lớn nhất** (Project Euler 14) — memo là bắt buộc khi `N = 10^6`.
2. **Sắp giảm dần theo power, hòa thì giá trị giảm dần:** `key=lambda x: (-power(x), -x)` hoặc `reverse=True`.
3. **Nhiều truy vấn `(lo, hi, k)`** trên cùng miền: tính bảng power một lần, mỗi truy vấn dùng heap hoặc quickselect.
4. **LeetCode 1356 — Sort Integers by The Number of 1 Bits:** cùng khuôn sắp theo khóa `(bin(x).count("1"), x)`.
5. **LeetCode 1636 — Sort Array by Increasing Frequency:** khóa `(freq[x], -x)`.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Trực tiếp + sort | mô phỏng dãy cho từng số | `O(N·P + N log N)` | `O(N)` | ràng buộc nhỏ, code ngắn nhất |
| 2. `lru_cache` + sort | memo đệ quy | `O(D + N log N)` | `O(D)` | nhanh để viết, có memo |
| 3. Memo lặp + sort | đi tới giá trị đã biết, điền ngược | `O(D + N log N)` | `O(D)` | dãy dài, tránh đệ quy; **khuyên dùng** |
| 4. Bảng sẵn + heap | tiền xử lý 1..1000, chọn `k` nhỏ nhất | `O(N log k)` mỗi truy vấn | `O(D)` | nhiều truy vấn, `k` nhỏ |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** power của `1`, `7`, `16`, `27` là bao nhiêu?

**Bài 2 (tính tay):** viết dãy Collatz của `6` và tính power(6). Sau đó cho biết `getKth(1, 6, 3)`.

**Bài 3 (điền TODO):** hoàn thiện hàm memo **lặp** sau.

In [25]:
def power_todo(x, memo):
    """Bài 3 — điền TODO. (Bài tập — chưa gọi)"""
    path = []
    # TODO: trong khi x chưa có trong memo: thêm x vào path, x = bước kế tiếp
    # TODO: p = memo[x]; duyệt path ngược, mỗi v: p += 1, memo[v] = p
    return memo.get(path[0] if path else x)

**Bài 4 (tự viết lại):** viết `kth_quickselect(lo, hi, k)` không sắp toàn bộ: dùng `heapq.nsmallest` **không** truyền `key` (gợi ý: tạo tuple `(power, x)` trước).

**Bài 5 (mini challenge):** viết `max_power_upto(N)` trả về cặp `(x, power)` với `x ≤ N` có power **lớn nhất** (hòa thì `x` nhỏ nhất). Chạy được với `N = 10^5` trong dưới 1 giây.

In [26]:
def max_power_upto(N):
    """Bài 5 — tự viết. (Bài tập — chưa gọi)"""
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | `power(1) = 0`; 16 là lũy thừa 2 | 7 đã tính ở mục 4 | 0, 16, 4, 111 |
| 2 | 6 → 3 → … | power(6) = 1 + power(3) | dãy `6,3,10,5,16,8,4,2,1`, power 8; power 1..6 = `0,1,7,2,5,8` ⇒ thứ tự `1,2,4,5,3,6` ⇒ 4 |
| 3 | giống Cách 3 | `while x not in memo` | điền ngược bằng `reversed(path)` |
| 4 | tuple so sánh tự nhiên | `heapq.nsmallest(k, [(P[x], x) for x in ...])` | lấy `[-1][1]` |
| 5 | memo dùng chung cho mọi `x` | duyệt `x = 1..N`, giữ max | so `(power, -x)` để hòa thì `x` nhỏ thắng |

## 19. Đáp án tham khảo cho bài tập

In [27]:
# Bài 1
ans1 = [power_direct(x) for x in (1, 7, 16, 27)]
print("Bài 1:", ans1)
assert ans1 == [0, 16, 4, 111]

# Bài 2
seq6 = [6]
while seq6[-1] != 1:
    v = seq6[-1]; seq6.append(v // 2 if v % 2 == 0 else 3 * v + 1)
print("Bài 2: dãy 6 =", seq6, "| power(6) =", len(seq6) - 1, "| getKth(1, 6, 3) =", kth_memo(1, 6, 3))
assert len(seq6) - 1 == 8 and kth_memo(1, 6, 3) == 4

Bài 1: [0, 16, 4, 111]
Bài 2: dãy 6 = [6, 3, 10, 5, 16, 8, 4, 2, 1] | power(6) = 8 | getKth(1, 6, 3) = 4


In [28]:
# Bài 3
def power_todo_sol(x, memo):
    path = []
    while x not in memo:
        path.append(x)
        x = x // 2 if x % 2 == 0 else 3 * x + 1
    p = memo[x]
    for v in reversed(path):
        p += 1
        memo[v] = p
    return p

m3 = {1: 0}
assert all(power_todo_sol(x, m3) == power_direct(x) for x in range(1, 1001))
assert power_todo_sol(27, m3) == 111
print("Bài 3: OK")

Bài 3: OK


In [29]:
# Bài 4
def kth_quickselect(lo, hi, k):
    return heapq.nsmallest(k, [(POWER_TABLE[x], x) for x in range(lo, hi + 1)])[-1][1]

rng = random.Random(4)
for _ in range(300):
    lo = rng.randint(1, 1000); hi = rng.randint(lo, 1000); k = rng.randint(1, hi - lo + 1)
    assert kth_quickselect(lo, hi, k) == kth_direct(lo, hi, k)
print("Bài 4: OK")

Bài 4: OK


In [30]:
# Bài 5
def max_power_upto_sol(N):
    memo = {1: 0}
    best = (0, -1)                      # (power, -x)
    for x in range(1, N + 1):
        p = power_todo_sol(x, memo)
        best = max(best, (p, -x))
    return -best[1], best[0]

assert max_power_upto_sol(10) == (9, 19)
assert max_power_upto_sol(1000) == (871, 178)
t0 = time.perf_counter()
r5 = max_power_upto_sol(100_000)
print("Bài 5:", r5, f"trong {time.perf_counter() - t0:.3f}s")
assert r5 == (77031, 350)

Bài 5: (77031, 350) trong 0.084s


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "tính đại lượng có memo rồi sắp theo khóa tuple":**

```python
memo = {1: 0}
def power(x):
    path = []
    while x not in memo:
        path.append(x)
        x = x // 2 if x % 2 == 0 else 3 * x + 1
    p = memo[x]
    for v in reversed(path):
        p += 1; memo[v] = p
    return p

return sorted(range(lo, hi + 1), key=lambda x: (power(x), x))[k - 1]
```

**Checklist trước khi submit:**
- [ ] `power(1) = 0`; đếm **bước**, không đếm số phần tử.
- [ ] Dùng `//` (chia nguyên), không `/`.
- [ ] Khóa sắp là `(power(x), x)` — đủ hai tiêu chí.
- [ ] `range(lo, hi + 1)` bao gồm `hi`; phần tử thứ `k` là `[k − 1]`.
- [ ] Memo bằng `dict` (giá trị trung gian vượt `hi`).
- [ ] Đúng chữ ký: `getKth(self, lo: int, hi: int, k: int) -> int`.